# How the feed gets cleaned: a walkthrough

This notebook runs the same steps the consumer does, one small step at a time, and prints what happens.
Press **Run All** (or run one cell at a time). It needs no database and no RabbitMQ.

**The idea:** the registry sends us 2,518 messages about 2,000 companies. We cannot trust them, so we
1. look at the data and find what is wrong,
2. clean each message (or set it aside with a reason),
3. combine the messages for each company into one correct record,
4. turn the description into numbers so we can find similar companies.

## Step 1. Open the file and look at one normal line

In [1]:
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

# Find the project folder, so this notebook works from any folder inside the project.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "similarco").exists())
sys.path.insert(0, str(ROOT))

lines = (ROOT / "data" / "feed.jsonl").read_text(encoding="utf-8").splitlines()
print("number of lines (messages) in the feed file:", len(lines))
print()
print("line 1, laid out so it is easy to read:")
print(json.dumps(json.loads(lines[0]), indent=2))

number of lines (messages) in the feed file: 2518

line 1, laid out so it is easy to read:
{
  "type": "company-upserted",
  "registrationNumber": "SYN-001777",
  "updatedAt": "2026-08-14T09:00:00+08:00",
  "data": {
    "companyName": "Global Maju Packaging Sdn. Bhd.",
    "natureOfBusiness": "Production of rubber gloves for export",
    "state": "Negeri Sembilan",
    "industry": "Manufacturing"
  }
}


In plain words, line 1 says: *"company SYN-001777 was added or changed at this time, and here are its details."*
Every line is supposed to look like this. Let's check whether they do.

## Step 2. Try to read every line

In [2]:
readable, broken, nulls = [], [], []
for number, text in enumerate(lines, start=1):
    try:
        message = json.loads(text)
    except json.JSONDecodeError:
        broken.append(number)
        continue
    if message is None:
        nulls.append(number)
    else:
        readable.append((number, message))

print("readable messages          :", len(readable))
print("broken (cut off) lines     :", len(broken), "-> line numbers", broken)
print("lines that are only 'null' :", len(nulls), "-> line numbers", nulls)
print()
print("what a broken line looks like (it just stops):")
print(lines[broken[0] - 1])

readable messages          : 2507
broken (cut off) lines     : 5 -> line numbers [98, 1387, 1561, 1850, 1883]
lines that are only 'null' : 6 -> line numbers [237, 358, 575, 807, 1972, 2299]

what a broken line looks like (it just stops):
{"type": "company-upserted", "registrationNumber": "SYN-000509", "updatedAt": "2026-08-14T09:15:13+08:00", "data": {"companyName": "Bes


## Step 3. Does every message have the same keys?

If the data were clean, every message would have the same four keys. Count how many different combinations exist.

In [3]:
key_combinations = Counter(tuple(sorted(message.keys())) for _, message in readable)
for keys, count in key_combinations.most_common():
    print(f"{count:5d} messages have these keys: {keys}")

 2405 messages have these keys: ('data', 'registrationNumber', 'type', 'updatedAt')
   38 messages have these keys: ('data', 'registrationNumber', 'source', 'type', 'updatedAt')
   37 messages have these keys: ('data', 'ingestedBy', 'registrationNumber', 'type', 'updatedAt')
   13 messages have these keys: ('registrationNumber', 'type', 'updatedAt')
    7 messages have these keys: ('data', 'registrationNumber', 'type')
    7 messages have these keys: ('data', 'type', 'updatedAt')


The first row is the normal case. The others are messages with something missing (`registrationNumber` or
`updatedAt`), or an extra key (`source`, `ingestedBy`).

## Step 4. What kind of value is the registration number?

In [4]:
def kind(value):
    return "missing" if value is None else type(value).__name__


print("kinds of registrationNumber:", dict(Counter(kind(m.get("registrationNumber")) for _, m in readable)))
print()


def shape(text):
    return re.sub(r"\d", "9", text)  # replace every digit with 9, so we see the pattern, not the number


shapes = Counter(shape(m["registrationNumber"]) for _, m in readable if isinstance(m.get("registrationNumber"), str))
print("the text ones come in these shapes (quotes added so you can see spaces):")
for pattern, count in shapes.most_common():
    print(f"{count:5d}   '{pattern}'")
print()
print("the 3 that are plain numbers:")
for number, m in readable:
    if isinstance(m.get("registrationNumber"), int):
        print(f"   line {number}: registrationNumber={m['registrationNumber']}, name={m['data']['companyName']!r}")

kinds of registrationNumber: {'str': 2497, 'missing': 7, 'int': 3}

the text ones come in these shapes (quotes added so you can see spaces):
 2448   'SYN-999999'
   27   ' SYN-999999 '
   22   'syn-999999'

the 3 that are plain numbers:
   line 1177: registrationNumber=119, name='X'
   line 1684: registrationNumber=1756, name='X'
   line 2118: registrationNumber=64, name='X'


Most are `SYN-999999`. Some are lowercase, some have extra spaces, and 3 are just a number with the company name "X".

## Step 5. What kind of value is the date?

In [5]:
def date_kind(value):
    if value is None:
        return "missing"
    if isinstance(value, (int, float)):
        return "a big number (epoch milliseconds)"
    if value.endswith("Z"):
        return "text ending in Z (UTC time)"
    if re.search(r"[+-]\d\d:\d\d$", value):
        return "text ending in +08:00 (Malaysia time)"
    return "text with no timezone at all"


for what, count in Counter(date_kind(m.get("updatedAt")) for _, m in readable).most_common():
    print(f"{count:5d}  {what}")

 2428  text ending in +08:00 (Malaysia time)
   30  a big number (epoch milliseconds)
   28  text ending in Z (UTC time)
   14  text with no timezone at all
    7  missing


Four ways to write a time. They all mean "a moment", but you cannot compare `1786671038000` with
`2026-08-14T09:06:19` until both are converted to the same clock.

## Step 6. Spelling of the state

In [6]:
states = Counter(m["data"].get("state") for _, m in readable if isinstance(m.get("data"), dict))
print("Kuala Lumpur is written in", {s: n for s, n in states.items() if s and s.casefold() == "kuala lumpur"})
real = {s.casefold() for s in states if s}
print()
print(f"{len([s for s in states if s])} different spellings in the file, but only {len(real)} real states")

Kuala Lumpur is written in {'Kuala Lumpur': 550, 'KUALA LUMPUR': 61, 'kuala lumpur': 47}

46 different spellings in the file, but only 16 real states


## Step 7. The same company, many messages

Group the messages by company. First we tidy the registration number with the project's real function
(`normalize_registration_number`), so ` SYN-001497 ` and `syn-001497` land in the same group.

In [7]:
from similarco.events import normalize_registration_number

by_company = defaultdict(list)
for number, message in readable:
    registration = normalize_registration_number(message.get("registrationNumber"))
    if registration:
        by_company[registration].append((number, message))

print("different companies:", len(by_company))
print("companies with more than one message:", sum(len(v) > 1 for v in by_company.values()))
print()
print("Look at one company, SYN-000675, in the order its messages arrived:")
for number, message in by_company["SYN-000675"]:
    print(f"   line {number:4d}   time {message['updatedAt']}   ->  {message['data']['natureOfBusiness']}")

different companies: 2000
companies with more than one message: 406

Look at one company, SYN-000675, in the order its messages arrived:
   line    8   time 2026-08-14T09:00:48+08:00   ->  Last-mile parcel delivery across Seremban
   line 2411   time 2026-08-14T20:38:48+08:00   ->  Cold chain transport and storage of frozen goods
   line 2413   time 2026-08-14T14:49:48+08:00   ->  Trucking services for construction materials


Look closely: the message that arrived **last** (line 2413) has the **earlier** time (14:49), and the newest one
(20:38) arrived before it. If we simply kept the last message to arrive, we would keep stale information.

## Step 8. The cleaning function, on the problem lines

`parse_event` is the real cleaning function from `similarco/events.py`. For each line it either returns a clean
event or says why it cannot be used. Watch what it does to the problem lines we found.

In [8]:
from similarco.events import Rejected, parse_event


def show(line_number):
    text = lines[line_number - 1]
    result = parse_event(text.encode())
    print(f"line {line_number}: {text[:95]}")
    if isinstance(result, Rejected):
        print(f"      -> REJECTED, reason: {result.reason}")
    else:
        when = f"{result.updated_at:%Y-%m-%d %H:%M:%S} UTC"
        print(f"      -> kept as {result.registration_number}, time {when}")
        print(f"         details: {result.field_values()}")
    print()


problems = {
    98: "cut off in the middle",
    237: "just the word null",
    132: "no registration number",
    1177: "registration number is a plain number",
    130: "lowercase registration number",
    157: "registration number with spaces",
    47: "time ends in Z",
    67: "time has no timezone",
    201: "time is a big number",
    56: "no time, almost no details",
    27: "state in CAPITALS",
    39: "description repeated hundreds of times",
}
for line_number, why in problems.items():
    print("#", why)
    show(line_number)

# cut off in the middle
line 98: {"type": "company-upserted", "registrationNumber": "SYN-000509", "updatedAt": "2026-08-14T09:15
      -> REJECTED, reason: invalid_json

# just the word null
line 237: null
      -> REJECTED, reason: not_an_object

# no registration number
line 132: {"type": "company-upserted", "updatedAt": "2026-08-14T09:20:07+08:00", "data": {"companyName": 
      -> REJECTED, reason: missing_registration_number

# registration number is a plain number
line 1177: {"type": "company-upserted", "registrationNumber": 119, "updatedAt": "2026-08-14T12:11:00+08:00
      -> REJECTED, reason: invalid_registration_number

# lowercase registration number
line 130: {"type": "company-upserted", "registrationNumber": "syn-001988", "updatedAt": "2026-08-14T09:15
      -> kept as SYN-001988, time 2026-08-14 01:15:01 UTC
         details: {'status': 'active', 'company_name': 'Gemilang Kirana Education Sdn. Bhd.', 'nature_of_business': 'Pusat tuisyen untuk pelajar sekolah rendah dan me

Notice what happened:
- Broken, `null`, no registration number and plain-number registration numbers were **rejected with a reason**,
  not guessed and not crashed on.
- Spaces, lowercase, capital letters, every time format and the repeated description were **fixed**.
- The message with no time was **kept** (the company must not be lost). It will simply lose against any message that has a time.

### The cleaning rules, in plain words

| Problem | What we do | Why |
|---|---|---|
| Line is cut off, or just `null` | set aside, reason recorded | there is nothing inside to use |
| No registration number | set aside | we cannot tell which company it is about |
| Registration number is a plain number (`119`) | set aside, **not guessed** | guessing `SYN-000119` could overwrite a real company with the name "X" |
| Lowercase or extra spaces (`syn-001988`) | tidy to `SYN-001988` | same company, different spelling |
| Four time formats | convert all to one clock (UTC) | to decide which message is newer they must be on the same clock |
| Time with no timezone | assume Malaysia time (+08:00) | every other time in the file is Malaysia time |
| No time at all | keep it, treat as the oldest possible | never lose a company, but never beat a message that has a time |
| State in CAPITALS | one spelling per state | so counting and filtering by state is right |
| Description repeated 100s of times | keep one copy | a long repeated text carries no extra meaning |
| A blank or missing field | means "no information", **not** "delete" | an empty message must not erase good data we already have |

## Step 9. Run the whole file through the cleaning

In [9]:
events = defaultdict(list)  # company -> its clean messages
rejected = Counter()
for text in lines:
    result = parse_event(text.encode())
    if isinstance(result, Rejected):
        rejected[str(result.reason)] += 1
    else:
        events[result.registration_number].append(result)

print("rejected, by reason:")
for reason, count in rejected.most_common():
    print(f"   {count:3d}  {reason}")
print("   ---")
print(f"   {sum(rejected.values()):3d}  rejected in total (of {len(lines)} lines)")
print()
print("companies kept:", len(events), " <- all 2,000 companies, none lost")

rejected, by reason:
     7  missing_registration_number
     6  not_an_object
     5  invalid_json
     3  invalid_registration_number
   ---
    21  rejected in total (of 2518 lines)

companies kept: 2000  <- all 2,000 companies, none lost


## Step 10. Combine many messages into one record

The rule: **for each detail, keep the value from the message with the newest time, not the one that arrived last.**
Here it is on SYN-000675.

In [10]:
import random

from similarco.merge import apply_events

group = events["SYN-000675"]
print("messages, in arrival order (times in UTC):")
for event in group:
    print(f"   {event.updated_at:%H:%M}   {event.nature_of_business}")
print()
print("if we kept the LAST ARRIVAL :", group[-1].nature_of_business, "  <- wrong, it is old")
print("our rule (newest time wins) :", apply_events(None, group).nature_of_business, "  <- right")

# Does the order of arrival matter? Shuffle the messages 1,000 times and see how many different answers we get.
random.seed(1)
answers = set()
for _ in range(1000):
    shuffled = group[:]
    random.shuffle(shuffled)
    answers.add(apply_events(None, shuffled).nature_of_business)
print()
print("answers after 1,000 random shuffles:", answers)

messages, in arrival order (times in UTC):
   01:00   Last-mile parcel delivery across Seremban
   12:38   Cold chain transport and storage of frozen goods
   06:49   Trucking services for construction materials

if we kept the LAST ARRIVAL : Trucking services for construction materials   <- wrong, it is old
our rule (newest time wins) : Cold chain transport and storage of frozen goods   <- right

answers after 1,000 random shuffles: {'Cold chain transport and storage of frozen goods'}


One answer, however the messages are shuffled. That is the property the tests check for every company in the file.

## Step 11. The finished records

In [11]:
final = {registration: apply_events(None, group) for registration, group in events.items()}
records = list(final.values())

print("companies                                    :", len(records))
print("struck off (kept, but never recommended)     :", sum(r.status == "struck_off" for r in records))
print("no description in ANY message (use industry):", sum(not r.nature_of_business for r in records))
print()
for registration in ["SYN-001550", "SYN-000675"]:
    r = final[registration]
    print(f"{r.registration_number}  {r.company_name}")
    print(f"    state={r.state}  industry={r.industry}  status={r.status}")
    print(f"    description={r.nature_of_business!r}")

companies                                    : 2000
struck off (kept, but never recommended)     : 8
no description in ANY message (use industry): 39

SYN-001550  Sinar Teguh Bakery Sdn. Bhd.
    state=Selangor  industry=Food & Beverage  status=active
    description='Operating cafes and coffee shops in the Klang Valley'
SYN-000675  Sejahtera Cahaya Cargo PLT
    state=Negeri Sembilan  industry=Logistics  status=active
    description='Cold chain transport and storage of frozen goods'


## Step 12. Print the clean data

This is the result of Steps 8 to 11: **one clean record per company**, with the spelling fixed, one registration
number format, and the newest value of every detail. First 15 companies, in registration-number order.

In [12]:
def cut(value, width):
    value = "-" if value is None else str(value)
    return value if len(value) <= width else value[: width - 1] + "…"


def print_table(rows, columns, widths):
    header = " | ".join(f"{name:<{w}}" for name, w in zip(columns, widths, strict=True))
    print(header)
    print("-" * len(header))
    for row in rows:
        print(" | ".join(f"{cut(value, w):<{w}}" for value, w in zip(row, widths, strict=True)))


COLUMNS = ["registration", "company name", "state", "industry", "status", "business description"]
WIDTHS = [12, 32, 15, 21, 10, 46]


def clean_row(r):
    description = r.nature_of_business or "(none: will use industry + name)"
    return [r.registration_number, r.company_name, r.state, r.industry, r.status, description]


ordered = sorted(records, key=lambda r: r.registration_number)
print("THE FIRST 15 CLEAN COMPANIES")
print_table([clean_row(r) for r in ordered[:15]], COLUMNS, WIDTHS)

THE FIRST 15 CLEAN COMPANIES
registration | company name                     | state           | industry              | status     | business description                          
-------------------------------------------------------------------------------------------------------------------------------------------------------
SYN-000001   | Murni Golden Runcit Trading      | Perak           | Retail                | active     | Online retail of baby products and toys       
SYN-000002   | Delima Pacific Kopi PLT          | Selangor        | -                     | active     | Restoran masakan Melayu dan Cina              
SYN-000003   | Global Bestari Runcit PLT        | Perak           | Retail                | active     | Retail of textiles and baju kurung            
SYN-000004   | Seri Unity Transport Sdn. Bhd.   | -               | Logistics             | active     | Trucking services for construction materials  
SYN-000005   | Unity Everbright Bakery PLT      | Pahang   

In [13]:
print("THE 8 STRUCK-OFF COMPANIES (kept, with their own page, but never recommended to others)")
print_table([clean_row(r) for r in ordered if r.status == "struck_off"], COLUMNS, WIDTHS)
print()
print("3 OF THE 39 COMPANIES THAT HAVE NO DESCRIPTION IN ANY MESSAGE (they use industry + name instead)")
print_table([clean_row(r) for r in ordered if not r.nature_of_business][:3], COLUMNS, WIDTHS)

THE 8 STRUCK-OFF COMPANIES (kept, with their own page, but never recommended to others)
registration | company name                     | state           | industry              | status     | business description                          
-------------------------------------------------------------------------------------------------------------------------------------------------------
SYN-000186   | Indah Prima Tani Enterprise      | Pahang          | Agriculture           | struck_off | Poultry farming and egg production            
SYN-000188   | Gemilang Gemilang Freight Sdn. … | Kuala Lumpur    | Logistics             | struck_off | Trucking services for construction materials  
SYN-000403   | Pacific Bestari Management Sdn.… | Johor           | Professional Services | struck_off | Management consulting for SMEs                
SYN-000561   | Mutiara Global Technologies Ent… | Kelantan        | Technology            | struck_off | Software development for small business accou…


In [14]:
for title, values in [("industry", [r.industry for r in records]), ("state", [r.state for r in records])]:
    print(f"companies by {title}, after cleaning (one spelling each):")
    for name, n in Counter(values).most_common():
        print(f"   {n:4d}  {name}")
    print()

companies by industry, after cleaning (one spelling each):
    189  Professional Services
    185  Agriculture
    182  Logistics
    179  Hospitality
    179  Technology
    175  Healthcare
    172  Retail
    167  Education
    163  Construction
    159  Manufacturing
    148  Food & Beverage
    102  None

companies by state, after cleaning (one spelling each):
    531  Kuala Lumpur
    449  Selangor
    178  Johor
    117  Pulau Pinang
     94  Sarawak
     93  Perak
     93  Kedah
     79  Sabah
     63  Negeri Sembilan
     58  Melaka
     49  None
     47  Pahang
     46  Terengganu
     37  Kelantan
     23  Perlis
     22  Labuan
     21  Putrajaya


## Step 13. Before and after

Each row below is one fix from Step 8, shown as the raw value on the left and the clean value on the right.

In [15]:
def before_after(line_number):
    text = lines[line_number - 1]
    raw = json.loads(text)
    event = parse_event(text.encode())
    data = raw["data"] if isinstance(raw.get("data"), dict) else {}
    changes = []
    if raw.get("registrationNumber") != event.registration_number:
        changes.append(("registration number", repr(raw.get("registrationNumber")), repr(event.registration_number)))
    raw_time = raw.get("updatedAt")
    if not (isinstance(raw_time, str) and raw_time.endswith("+08:00")):
        shown = "(missing)" if raw_time is None else repr(raw_time)
        clean_time = f"{event.updated_at:%Y-%m-%d %H:%M:%S} UTC"
        changes.append(("time", shown, clean_time + ("  (oldest possible)" if raw_time is None else "")))
    if data.get("state") != event.state:
        changes.append(("state", repr(data.get("state")), repr(event.state)))
    description = data.get("natureOfBusiness")
    if description != event.nature_of_business:
        shown = f"({len(description)} characters) {description[:28]}..."
        changes.append(("description", shown, repr(event.nature_of_business)))
    for what, before, after in changes:
        print(f"line {line_number:4d}  {what:<20}: {before}   ->   {after}")


for line_number in [130, 157, 47, 67, 201, 56, 27, 39]:
    before_after(line_number)

line  130  registration number : 'syn-001988'   ->   'SYN-001988'
line  130  state               : 'PULAU PINANG'   ->   'Pulau Pinang'
line  157  registration number : ' SYN-001497 '   ->   'SYN-001497'
line   47  time                : '2026-08-14T01:03:35Z'   ->   2026-08-14 01:03:35 UTC
line   67  time                : '2026-08-14T09:06:19'   ->   2026-08-14 01:06:19 UTC
line   67  state               : 'perak'   ->   'Perak'
line  201  time                : 1786671038000   ->   2026-08-14 01:30:38 UTC
line   56  time                : (missing)   ->   1970-01-01 00:00:00 UTC  (oldest possible)
line   27  state               : 'KUALA LUMPUR'   ->   'Kuala Lumpur'
line   39  description         : (1859 characters) Perunding pengurusan untuk P...   ->   'Perunding pengurusan untuk PKS'


## Step 14. What was set aside

These 21 messages could not be used. Nothing about them is guessed. Each is kept in the `rejected_messages`
table with its reason, so someone can look at them later.

In [16]:
shelf = []
for number, text in enumerate(lines, start=1):
    result = parse_event(text.encode())
    if isinstance(result, Rejected):
        shelf.append((number, str(result.reason), text))

print(f"{len(shelf)} MESSAGES SET ASIDE")
print_table(shelf, ["line", "reason", "what the line looks like"], [5, 29, 84])

21 MESSAGES SET ASIDE
line  | reason                        | what the line looks like                                                            
----------------------------------------------------------------------------------------------------------------------------
98    | invalid_json                  | {"type": "company-upserted", "registrationNumber": "SYN-000509", "updatedAt": "2026…
132   | missing_registration_number   | {"type": "company-upserted", "updatedAt": "2026-08-14T09:20:07+08:00", "data": {"co…
237   | not_an_object                 | null                                                                                
345   | missing_registration_number   | {"type": "company-upserted", "updatedAt": "2026-08-14T09:56:07+08:00", "data": {"co…
358   | not_an_object                 | null                                                                                
575   | not_an_object                 | null                                                           

## Step 15. Save the clean data to a file

The same clean data, all 2,000 companies, saved as `clean_companies.csv` next to this notebook. You can open it
in VS Code or Excel. The last column says what the model will read for each company: the business `description`,
or a `fallback` made from industry and name when there is no description.

In [17]:
import csv

from similarco.embedding import embedding_input

target = ROOT / "notebooks" / "clean_companies.csv"
header = ["registration_number", "company_name", "state", "industry", "status", "nature_of_business", "embedding_basis"]
with target.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(header)
    for r in ordered:
        wanted = embedding_input(
            nature_of_business=r.nature_of_business, industry=r.industry, company_name=r.company_name
        )
        row = [r.registration_number, r.company_name, r.state, r.industry, r.status, r.nature_of_business]
        writer.writerow([*row, wanted.basis if wanted else ""])

print("saved", len(ordered), "clean companies to", target.relative_to(ROOT))
print()
print("the first 4 lines of that file:")
print(*target.read_text(encoding="utf-8").splitlines()[:4], sep=chr(10))

saved 2000 clean companies to notebooks/clean_companies.csv

the first 4 lines of that file:
registration_number,company_name,state,industry,status,nature_of_business,embedding_basis
SYN-000001,Murni Golden Runcit Trading,Perak,Retail,active,Online retail of baby products and toys,description
SYN-000002,Delima Pacific Kopi PLT,Selangor,,active,Restoran masakan Melayu dan Cina,description
SYN-000003,Global Bestari Runcit PLT,Perak,Retail,active,Retail of textiles and baju kurung,description


## Step 16. Turning a description into numbers

A computer cannot see that "coffee shop" and "kedai kopi" mean the same thing. So a model turns each sentence
into 768 numbers, a kind of "map position" for its meaning. Similar meaning, nearby position.
(This loads the model from the `.cache` folder; the first time ever it downloads about 1 GB.)
The model is `paraphrase-multilingual-mpnet-base-v2`, chosen after testing eight models (see DESIGN.md, decision 5).

In [18]:
import numpy as np

from similarco.config import Settings
from similarco.embedding import FastEmbedder

embedder = FastEmbedder(cache_dir=ROOT / Settings.from_env().model_cache_dir)
phrases = [
    "Operating cafes and coffee shops",
    "Mengusahakan kedai kopi dan kafe",  # the same thing, in Malay
    "Production of rubber gloves for export",
]
vectors = np.array(embedder.embed(phrases))

print("each sentence became", vectors.shape[1], "numbers. The first 5 of sentence 1:", vectors[0][:5].round(3))
print()
print("similarity: 1.0 means identical meaning, near 0 means unrelated")
print("  coffee (English)  vs  coffee (Malay)    :", round(float(vectors[0] @ vectors[1]), 3))
print("  coffee (English)  vs  rubber gloves     :", round(float(vectors[0] @ vectors[2]), 3))
print("  coffee (Malay)    vs  rubber gloves     :", round(float(vectors[1] @ vectors[2]), 3))

each sentence became 768 numbers. The first 5 of sentence 1: [ 0.003 -0.011 -0.002  0.007 -0.   ]

similarity: 1.0 means identical meaning, near 0 means unrelated
  coffee (English)  vs  coffee (Malay)    : 0.92
  coffee (English)  vs  rubber gloves     : 0.147
  coffee (Malay)    vs  rubber gloves     : 0.135


## Step 17. "Similar companies" for one company

This is what the API does, here done in memory. We rank every other active company by how close its
description is to SYN-001550's description.

In [19]:
me = final["SYN-001550"]
texts = sorted({r.nature_of_business for r in records if r.nature_of_business})
vector_of = dict(zip(texts, embedder.embed(texts), strict=True))

companies_by_text = defaultdict(list)
for r in records:
    if r.registration_number != me.registration_number and r.status == "active" and r.nature_of_business:
        companies_by_text[r.nature_of_business].append(r)

ranked = sorted(companies_by_text, key=lambda text: -float(vector_of[me.nature_of_business] @ vector_of[text]))

print(f"{me.company_name}  |  {me.state}  |  {me.nature_of_business!r}")
print()
print("closest descriptions (similarity first), and how many companies have each one:")
for text in ranked[:6]:
    score = float(vector_of[me.nature_of_business] @ vector_of[text])
    print(f"   {score:.3f}   {text!r}   -> companies with this description: {len(companies_by_text[text])}")

Sinar Teguh Bakery Sdn. Bhd.  |  Selangor  |  'Operating cafes and coffee shops in the Klang Valley'

closest descriptions (similarity first), and how many companies have each one:
   1.000   'Operating cafes and coffee shops in the Klang Valley'   -> companies with this description: 1
   0.963   'Mengusahakan kedai kopi dan kafe di the Klang Valley'   -> companies with this description: 2
   0.891   'Operating cafes and coffee shops in Kuantan'   -> companies with this description: 3
   0.883   'Mengusahakan kedai kopi dan kafe di Kuching'   -> companies with this description: 2
   0.876   'Operating cafes and coffee shops in Kota Kinabalu'   -> companies with this description: 1
   0.874   'Mengusahakan kedai kopi dan kafe di Kuantan'   -> companies with this description: 1


Many companies share the same description, so many are *equally* similar. The API picks among them with a
stable shuffle so the same page always shows the same companies. In the real service all of this happens in
Postgres with pgvector, not in Python.

## What you just saw

1. We **looked** before cleaning: counting keys, kinds of values and spellings revealed every problem.
2. We **cleaned** each message: fixed what was fixable and set the rest aside with a reason.
3. We **combined** messages with one rule (newest time wins, per detail), which gives the same answer in any order.
4. We printed the **clean data**: 2,000 companies, one tidy record each, also saved as `clean_companies.csv`.
5. We turned descriptions into **numbers** and ranked by closeness.